# Assignment 2: Image Processing and Pyramid Blending

**Individual · 100 points**

This assignment investigates how filtering, dynamic-range transformations, multi-scale representations, and blending improve visual data.

## 0. Student Information

- First name: Bryan
- Last name: Palomo
- ABC123: tkq516
- External resources and generative-AI assistance: cv/numpy docs, claude for distribution functions and researching pyramids

This is an individual assignment. Do not install unapproved packages or use external libraries or pretrained pipelines to bypass a required implementation.

Generative AI tools may be used for conceptual clarification, debugging assistance, code suggestions, and help interpreting documentation. Students must disclose meaningful use in every assignment. You remain responsible for understanding, testing, and explaining all submitted work. AI may not replace your own experimental analysis, failure analysis, reflection, or interpretation of observed results. You may be asked to explain selected work; inability to do so may reduce credit or lead to an academic-integrity review.


## 1. Assignment Overview and Learning Objectives

By completing A2, you should be able to implement and analyze spatial and nonlinear filtering; compare denoising under distinct noise models; manipulate contrast and dynamic range; construct and reconstruct Gaussian and Laplacian pyramids; perform Laplacian pyramid blending; evaluate results with visual and quantitative evidence; and diagnose limitations.

**Point map:** core implementation 40; required experiments and evidence 21; pyramid decomposition 6; analysis and interpretation 18 (10 per-experiment responses, 3 synthesis, 5 failure analysis); reproducibility, organization, and submission compliance 10; extension 5. The six categories sum to 100.

| Tables & Metadata | Figures | Other Outputs |
|---|---|---|
| `experiment_metrics.csv` | `filtering_results.png` | one declared extension artifact |
| `pyramid_metrics.json` | `denoising_comparison.png` | |
| | `contrast_enhancement.png` | |
| | `pyramid_visualization.png` | |
| | `pyramid_reconstruction.png` | |
| | `blending_comparison.png` | |
| | `pyramid_decomposition.png` | |
| | `failure_analysis.png` | |


## 2. Environment Verification

Run this setup first. It discovers the repository independently of the notebook working directory and verifies the immutable assets.


In [1]:
from pathlib import Path
import json, sys
import cv2
import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio, structural_similarity
import cs5243
from cs5243.data import find_course_root
COURSE_ROOT=find_course_root(Path(cs5243.__file__))
A2=COURSE_ROOT/"A2"; sys.path.insert(0,str(A2/"src"))
import a2_tools
import student_code as sc
PATHS=a2_tools.ensure_output_directories()
problems=a2_tools.verify_asset_hashes(); assert not problems, problems
print({"assignment":"A2-2026.1","environment":"2026.1"})


{'assignment': 'A2-2026.1', 'environment': '2026.1'}


## 3. Dataset and Problem Setup

The dataset contains odd dimensions, strong edges, gradients, thin structure, texture, controlled Gaussian and impulse noise, four aligned blend pairs spanning five mask geometries (a soft vertical seam, a circular spot, and a diagonal seam over synthetic scenes, plus a hard-edged vertical seam and a hard-edged circular seam over real fruit photographs), an underexposed contrast source, and a high-frequency difficult case. Most assets are generated programmatically (CC0 1.0); the two real-photo blend pairs are retained course-legacy teaching images credited in `data/ATTRIBUTION.md`. Unlike the synthetic masks, the real-photo masks are hard-edged rather than feathered, so direct blending (splicing) leaves an obvious seam that Laplacian pyramid blending visibly smooths. Do not edit the assets.

Implement reusable graded functions in `src/student_code.py`; use the notebook to call them, conduct experiments, save required artifacts, and explain results. Maintain one cumulative `outputs/tables/experiment_metrics.csv` throughout the notebook using the documented column names, and append later finite-valued records without deleting earlier experiments.

In [2]:
images=a2_tools.load_images()
print({name:(value.shape,str(value.dtype)) for name,value in images.items()})


{'blend2_left.png': ((257, 385, 3), 'uint8'), 'blend2_mask.png': ((257, 385), 'uint8'), 'blend2_right.png': ((257, 385, 3), 'uint8'), 'blend3_left.png': ((257, 385, 3), 'uint8'), 'blend3_mask.png': ((257, 385), 'uint8'), 'blend3_right.png': ((257, 385, 3), 'uint8'), 'blend4_left.png': ((257, 385, 3), 'uint8'), 'blend4_mask.png': ((257, 385), 'uint8'), 'blend4_right.png': ((257, 385, 3), 'uint8'), 'blend_left.png': ((257, 385, 3), 'uint8'), 'blend_mask.png': ((257, 385), 'uint8'), 'blend_mask_spot.png': ((257, 385), 'uint8'), 'blend_right.png': ((257, 385, 3), 'uint8'), 'color_scene.png': ((257, 385, 3), 'uint8'), 'difficult_high_frequency.png': ((193, 319), 'uint8'), 'exposure_under.png': ((257, 385, 3), 'uint8'), 'gaussian_noisy.png': ((255, 383), 'uint8'), 'impulse_noisy.png': ((255, 383), 'uint8'), 'test_pattern_gray.png': ((255, 383), 'uint8')}


## 4. Core Implementation

### Task 1 — convolution, Gaussian kernels, and sharpening (10 implementation + 5 evidence points)

Implement `convolve2d_manual`, `gaussian_kernel`, and `sharpen_image` to their contracts. `convolve2d_manual` is true convolution, including spatial kernel reversal; it must not call a library convolution/correlation routine. Compare your filtering with an explicitly identified library reference only in the experiment. Preserve numeric values until display or saving. Save `outputs/figures/filtering_results.png` and append filtering/reference-error evidence to `experiment_metrics.csv`.

Use `test_pattern_gray.png` (converted with `a2_tools.to_float01`) as the filtering source; `color_scene.png` is available if you also want to demonstrate multichannel support. Any OpenCV filtering routine is an acceptable library reference as long as you name it in the notebook and match its border handling to the mode you pass your own function. You choose the kernel sizes, sigmas, border modes, sharpening amounts, and error metric: that design is part of the evidence and analysis marks, so vary them enough to support the claims the analysis prompt asks for.


In [3]:
gray = a2_tools.to_float01(images["test_pattern_gray.png"])

METRICS_PATH = PATHS["tables"]/"experiment_metrics.csv"

def append_metrics(rows):
    df_new = pd.DataFrame(rows)
    if METRICS_PATH.exists():
        df_old = pd.read_csv(METRICS_PATH)
        df = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df = df_new
    df.to_csv(METRICS_PATH, index=False)

kernel_configs = [(5, 1.0), (9, 2.0), (15, 3.0)]

blur_results = {}
cv_blur_results = {}
for size, sigma in kernel_configs:
    kernel = sc.gaussian_kernel(size, sigma)
    blur = sc.convolve2d_manual(gray, kernel, border_mode="reflect")
    cv_blur = cv2.GaussianBlur(gray, (size, size), sigma, borderType=cv2.BORDER_REFLECT_101)
    blur_results[(size, sigma)] = blur
    cv_blur_results[(size, sigma)] = cv_blur
    err = float(np.mean(np.abs(blur - cv_blur)))
    append_metrics([{"experiment": "filtering", "method": "convolve2d_manual_vs_cv2.GaussianBlur",
                      "condition": f"size={size}_sigma={sigma}", "metric": "mean_abs_error",
                      "value": err, "units": "intensity"}])

sharpen_amounts = [0.0, 1.0, 2.0, 4.0]
sharpen_results = {}
for amount in sharpen_amounts:
    sharp = sc.sharpen_image(gray, amount, 1.2)
    sharpen_results[amount] = sharp
    overshoot = float(np.mean((sharp > 1.0) | (sharp < 0.0)))
    append_metrics([{"experiment": "filtering", "method": "sharpen_image",
                      "condition": f"amount={amount}_sigma=1.2", "metric": "out_of_range_fraction",
                      "value": overshoot, "units": "fraction"}])

fig, axes = a2_tools.axes_grid(1, 4)
axes[0, 0].imshow(gray, cmap="gray"); axes[0, 0].set_title("original")
axes[0, 1].imshow(blur_results[(9, 2.0)], cmap="gray"); axes[0, 1].set_title("convolve2d_manual")
axes[0, 2].imshow(cv_blur_results[(9, 2.0)], cmap="gray"); axes[0, 2].set_title("cv2.GaussianBlur")
axes[0, 3].imshow(np.clip(sharpen_results[2.0], 0, 1), cmap="gray"); axes[0, 3].set_title("sharpen amount=2.0")
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()
fig.savefig(PATHS["figures"]/"filtering_results.png", dpi=150)
plt.close(fig)


**Filtering analysis (3–5 sentences):** Connect kernel support and sigma to smoothing, edge behavior, detail loss, halo formation, and measured evidence.
Larger sigma and kernel size do stronger smoothing on larger regions and less on finer details. Sharpening amount does the inverse. It amplifies high frequency content the higher amount. Manual convolution when compared with cv2 gaussian blur is within 3.2e-8 mean absolute intensity across the three configurations.



### Task 2 — noise models and denoising (8 implementation + 5 evidence points)

Implement deterministic Gaussian and impulse noise plus `median_filter_manual`. With declared seeds, use your two noise functions to create the observations used in the comparison. The distributed `gaussian_noisy.png` and `impulse_noisy.png` files are independent observations of `test_pattern_gray.png`, drawn with Gaussian sigma 0.08 and impulse probability 0.07; use `test_pattern_gray.png` as the clean reference, and use those parameter values unless you declare your own.

They are independent draws, so use them for visual comparison only. They are not bit-identical to your own output, so do not spend time trying to reproduce them exactly. Compare Gaussian, median, and OpenCV bilateral filtering on both noise types, using the same clean reference. Save `outputs/figures/denoising_comparison.png` and append MSE or RMSE, PSNR, and SSIM records to `experiment_metrics.csv`. Metrics supplement rather than replace visual evidence.


In [4]:
clean = a2_tools.to_float01(images["test_pattern_gray.png"])
rng_gauss = np.random.default_rng(42)
rng_impulse = np.random.default_rng(43)
gauss_noisy = sc.add_gaussian_noise(clean, 0.08, rng_gauss)
impulse_noisy = sc.add_impulse_noise(clean, 0.07, rng_impulse)

gauss_kernel = sc.gaussian_kernel(5, 1.0)

def denoise_variants(noisy):
    return {
        "gaussian_blur": sc.convolve2d_manual(noisy, gauss_kernel, border_mode="reflect"),
        "median": sc.median_filter_manual(noisy, 5, border_mode="reflect"),
        "bilateral": cv2.bilateralFilter(noisy.astype(np.float32), 9, 0.15, 5),
    }

gauss_denoised = denoise_variants(gauss_noisy)
impulse_denoised = denoise_variants(impulse_noisy)

def metrics_rows(experiment, method, condition, clean_ref, estimate):
    mse = float(mean_squared_error(clean_ref, estimate))
    rmse = float(np.sqrt(mse))
    psnr = float(peak_signal_noise_ratio(clean_ref, estimate, data_range=1.0))
    ssim = float(structural_similarity(clean_ref, estimate, data_range=1.0))
    return [
        {"experiment": experiment, "method": method, "condition": condition, "metric": "mse", "value": mse, "units": "intensity^2"},
        {"experiment": experiment, "method": method, "condition": condition, "metric": "rmse", "value": rmse, "units": "intensity"},
        {"experiment": experiment, "method": method, "condition": condition, "metric": "psnr", "value": psnr, "units": "dB"},
        {"experiment": experiment, "method": method, "condition": condition, "metric": "ssim", "value": ssim, "units": "ratio"},
    ]

for method, estimate in gauss_denoised.items():
    append_metrics(metrics_rows("denoising", method, "gaussian_noise", clean, np.clip(estimate, 0, 1)))
for method, estimate in impulse_denoised.items():
    append_metrics(metrics_rows("denoising", method, "impulse_noise", clean, np.clip(estimate, 0, 1)))

fig, axes = a2_tools.axes_grid(2, 5)
axes[0, 0].imshow(clean, cmap="gray"); axes[0, 0].set_title("clean")
axes[0, 1].imshow(gauss_noisy, cmap="gray"); axes[0, 1].set_title("gaussian noisy")
for ax, (method, estimate) in zip(axes[0, 2:], gauss_denoised.items()):
    ax.imshow(np.clip(estimate, 0, 1), cmap="gray"); ax.set_title(method)
axes[1, 0].imshow(clean, cmap="gray"); axes[1, 0].set_title("clean")
axes[1, 1].imshow(impulse_noisy, cmap="gray"); axes[1, 1].set_title("impulse noisy")
for ax, (method, estimate) in zip(axes[1, 2:], impulse_denoised.items()):
    ax.imshow(np.clip(estimate, 0, 1), cmap="gray"); ax.set_title(method)
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()
fig.savefig(PATHS["figures"]/"denoising_comparison.png", dpi=150)
plt.close(fig)


**Denoising analysis (3–5 sentences):** Explain why rankings differ by noise model and identify a detail-preservation tradeoff.
For gaussian noise, bilateral performed best. Gaussian blur is small and localized so local averaging reduces it well with its edge aware weighting. For salt and pepper noise median filtering does best. Impulse replaces pixels with extreme outliers where averaging is weak and median is stronger. So a tradeoff present is that average filtering preserves fine gradients but is sensitive to outliers while median filtering does better with outliers but loses high frequency information more.


### Task 3 — contrast and dynamic range (7 implementation + 4 evidence points)

Implement percentile normalization, manual grayscale histogram equalization, and gamma correction. Compare at least three methods or settings on `exposure_under.png`. Address clipping, tonal distribution, noise amplification, and luminance-versus-channel processing. Save `outputs/figures/contrast_enhancement.png` and append measurements to the cumulative metrics file. OpenCV CLAHE may be a comparison but not a substitute for required work.


In [5]:
under = a2_tools.to_float01(images["exposure_under.png"])
gray_u8 = cv2.cvtColor((under*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)

stretched = sc.normalize_contrast(under, 2, 98)
gamma_bright = sc.apply_gamma(under, 0.5)
gamma_dark = sc.apply_gamma(under, 1.5)
equalized_gray = sc.histogram_equalize_gray(gray_u8)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clahe_result = clahe.apply(gray_u8)

def clipped_fraction(image, lo=0.0, hi=1.0):
    return float(np.mean((image <= lo) | (image >= hi)))

for name, result in [("normalize_contrast", stretched), ("gamma_0.5", gamma_bright), ("gamma_1.5", gamma_dark)]:
    append_metrics([
        {"experiment": "contrast", "method": name, "condition": "exposure_under", "metric": "mean_intensity", "value": float(result.mean()), "units": "intensity"},
        {"experiment": "contrast", "method": name, "condition": "exposure_under", "metric": "clipped_fraction", "value": clipped_fraction(result), "units": "fraction"},
    ])
append_metrics([
    {"experiment": "contrast", "method": "histogram_equalize_gray", "condition": "exposure_under_gray", "metric": "mean_intensity", "value": float(equalized_gray.mean() / 255.0), "units": "intensity"},
])

fig, axes = a2_tools.axes_grid(2, 3)
axes[0, 0].imshow(under); axes[0, 0].set_title("original")
axes[0, 1].imshow(np.clip(stretched, 0, 1)); axes[0, 1].set_title("normalize_contrast")
axes[0, 2].imshow(np.clip(gamma_bright, 0, 1)); axes[0, 2].set_title("gamma=0.5")
axes[1, 0].imshow(np.clip(gamma_dark, 0, 1)); axes[1, 0].set_title("gamma=1.5")
axes[1, 1].imshow(equalized_gray, cmap="gray"); axes[1, 1].set_title("histogram_equalize_gray")
axes[1, 2].imshow(clahe_result, cmap="gray"); axes[1, 2].set_title("cv2 CLAHE (reference)")
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()
fig.savefig(PATHS["figures"]/"contrast_enhancement.png", dpi=150)
plt.close(fig)


**Contrast analysis (3–5 sentences):** Tie visible changes to clipped fractions or tonal measurements and state one limitation.
Compared to cv CLAHE for reference. Percentile normalization has more intense contrasting but clips quite a bit resulting in diminished bright and dark colors. Gamma is more range safe as it doesn't result in clipping but scales values non-linearly. Histogram was done on greyscale because it shifts the colors around if trying to apply to color channels. All these methods are global and not localized like clahe which can improve contrast on local regions


### Task 4 — Gaussian and Laplacian pyramids (8 implementation + 4 evidence points)

Implement all three pyramid functions with finest-to-coarsest ordering, multi-channel support, and correct reconstruction for odd dimensions. Approved resizing operations are permitted; pyramid logic remains student work. Build the saved artifacts from `color_scene.png` with `levels = 4`, and record that file name as `source_image` alongside `levels` in `pyramid_metrics.json`; you may show additional images as well. Save `pyramid_visualization.png`, `pyramid_reconstruction.png`, and `pyramid_metrics.json` containing the required reconstruction fields documented in the README.


In [6]:
scene = a2_tools.to_float01(images["color_scene.png"])
levels = 4
gpyr = sc.gaussian_pyramid(scene, levels)
lpyr = sc.laplacian_pyramid(scene, levels)
reconstructed = sc.reconstruct_laplacian_pyramid(lpyr)

diff = np.abs(reconstructed - scene)
max_err = float(diff.max())
mean_err = float(diff.mean())

pyramid_metrics = {
    "levels": levels,
    "level_shapes": [list(level.shape) for level in gpyr],
    "reconstruction": {
        "max_abs_error": max_err,
        "mean_abs_error": mean_err,
        "dtype": str(reconstructed.dtype),
    },
    "source_image": "color_scene.png",
}
a2_tools.save_json(PATHS["tables"]/"pyramid_metrics.json", pyramid_metrics)

append_metrics([
    {"experiment": "pyramid", "method": "reconstruct_laplacian_pyramid", "condition": "color_scene_levels4", "metric": "max_abs_error", "value": max_err, "units": "intensity"},
    {"experiment": "pyramid", "method": "reconstruct_laplacian_pyramid", "condition": "color_scene_levels4", "metric": "mean_abs_error", "value": mean_err, "units": "intensity"},
])

def rescale_laplacian(level):
    magnitude = np.abs(level)
    scale = magnitude.max(axis=(0, 1), keepdims=True) if level.ndim == 3 else magnitude.max()
    return np.clip(level / (scale + 1e-8) * 0.5 + 0.5, 0, 1)

fig, axes = a2_tools.axes_grid(2, levels)
for i, level in enumerate(gpyr):
    axes[0, i].imshow(np.clip(level, 0, 1)); axes[0, i].set_title(f"gaussian L{i}")
for i, level in enumerate(lpyr):
    axes[1, i].imshow(rescale_laplacian(level)); axes[1, i].set_title(f"laplacian L{i}")
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()
fig.savefig(PATHS["figures"]/"pyramid_visualization.png", dpi=150)
plt.close(fig)

fig, axes = a2_tools.axes_grid(1, 3)
axes[0, 0].imshow(scene); axes[0, 0].set_title("original")
axes[0, 1].imshow(np.clip(reconstructed, 0, 1)); axes[0, 1].set_title("reconstructed")
axes[0, 2].imshow(diff / (diff.max() + 1e-8)); axes[0, 2].set_title("abs diff (scaled)")
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()
fig.savefig(PATHS["figures"]/"pyramid_reconstruction.png", dpi=150)
plt.close(fig)


**Pyramid analysis (3–5 sentences):** Explain what different levels represent and use reconstruction error to distinguish mathematical correctness from visual plausibility.
Each level is the same picture scaled down with blur. Level 0 is full scale with each level after that being about half the size. Each laplacian level is the lost data from scaling down to the next level. It's the lost detail. When I reconstructed the image using the laplacian it was near identical to the original image with the biggest delta being 1.5e-8, which is indiscernible to the eye but is not mutually inclusive.


## 5. Required Experiments

### Task 5 — direct blending and Laplacian pyramid blending across multiple pairs (7 implementation + 3 evidence points)

Implement `laplacian_pyramid_blend`, where mask 0 selects `image_a`, mask 1 selects `image_b`, and intermediate values blend. The dataset provides four aligned blend pairs spanning five mask geometries: `blend_left`/`blend_right` with `blend_mask` (soft vertical seam) or `blend_mask_spot` (circular seam), `blend2_left`/`blend2_right` with `blend2_mask` (diagonal seam), `blend3_left`/`blend3_right` (real apple/orange photographs) with `blend3_mask` (hard vertical seam), and `blend4_left`/`blend4_right` (a second real apple/orange close-up pair) with `blend4_mask` (hard circular seam). The two real-photo masks are not feathered, so they give the clearest evidence of the difference between the two methods. Apply `laplacian_pyramid_blend` to at least two different image pairs (two mask geometries over the same pair does not count as two), comparing each against direct blending (splicing) using the same mask, and include one deliberately poor mask or configuration (for example, too few pyramid levels or a mask misaligned with the seam) on at least one pair. Save one combined `outputs/figures/blending_comparison.png` arranging results across the pairs you used, and append per-pair seam or transition evidence to the cumulative metrics file.

In [7]:
def direct_blend(image_a, image_b, mask):
    m = mask[..., None] if image_a.ndim == 3 else mask
    return image_a * (1.0 - m) + image_b * m

blend_pairs = [
    ("blend", "blend_left.png", "blend_right.png", "blend_mask.png", 5),
    ("blend3", "blend3_left.png", "blend3_right.png", "blend3_mask.png", 5),
]

blend_results = {}
for name, left_key, right_key, mask_key, levels in blend_pairs:
    left = a2_tools.to_float01(images[left_key])
    right = a2_tools.to_float01(images[right_key])
    mask = a2_tools.to_float01(images[mask_key])
    if mask.ndim == 3:
        mask = mask[..., 0]
    direct = direct_blend(left, right, mask)
    laplacian = sc.laplacian_pyramid_blend(left, right, mask, levels)
    seam_col = mask.shape[1] // 2
    direct_grad = float(np.mean(np.abs(np.diff(direct[:, seam_col-2:seam_col+2], axis=1))))
    laplacian_grad = float(np.mean(np.abs(np.diff(laplacian[:, seam_col-2:seam_col+2], axis=1))))
    append_metrics([
        {"experiment": "blending", "method": "direct_blend", "condition": name, "metric": "seam_gradient", "value": direct_grad, "units": "intensity"},
        {"experiment": "blending", "method": "laplacian_pyramid_blend", "condition": name, "metric": "seam_gradient", "value": laplacian_grad, "units": "intensity"},
    ])
    blend_results[name] = (left, right, mask, direct, laplacian)

poor_levels = 1
left, right, mask, _, _ = blend_results["blend3"]
poor_blend = sc.laplacian_pyramid_blend(left, right, mask, poor_levels)
seam_col = mask.shape[1] // 2
poor_grad = float(np.mean(np.abs(np.diff(poor_blend[:, seam_col-2:seam_col+2], axis=1))))
append_metrics([
    {"experiment": "blending", "method": "laplacian_pyramid_blend", "condition": "blend3_poor_levels1", "metric": "seam_gradient", "value": poor_grad, "units": "intensity"},
])

fig, axes = a2_tools.axes_grid(2, 4)
for row, name in enumerate(["blend", "blend3"]):
    left, right, mask, direct, laplacian = blend_results[name]
    axes[row, 0].imshow(mask, cmap="gray"); axes[row, 0].set_title(f"{name} mask")
    axes[row, 1].imshow(np.clip(direct, 0, 1)); axes[row, 1].set_title("direct blend")
    axes[row, 2].imshow(np.clip(laplacian, 0, 1)); axes[row, 2].set_title("laplacian blend")
axes[0, 3].imshow(np.clip(poor_blend, 0, 1)); axes[0, 3].set_title("laplacian levels=1 (poor)")
axes[1, 3].axis("off")
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()
fig.savefig(PATHS["figures"]/"blending_comparison.png", dpi=150)
plt.close(fig)


**Blending analysis (3–5 sentences):** Discuss seam visibility, low- and high-frequency transitions, mask design, and boundary artifacts.
On the soft seam synthetic pair of images, the pyramid blending has smaller change at the seam vs just splicing the images together. On the real images, mathematically, the seam had a more significant change while being more smooth visually. Coarse, low frequency content blends over a wider region while fine high frequency content blends right at the seam.


### Task 6 — pyramid decomposition visualization (4 evidence + 2 analysis points)

For one synthetic pair and one real-photo pair (state which you used), use your already-implemented `gaussian_pyramid`, `laplacian_pyramid`, and `laplacian_pyramid_blend` to make the blending process itself visible instead of only its result. For each pair, show: the Gaussian pyramid of each source image; the Laplacian pyramid of each source image, rescaled per level so the band-pass structure is visible instead of near-black; the blended Laplacian pyramid produced by combining each level with the mask's Gaussian pyramid; and the progressive reconstruction from the coarsest level up to the final full-resolution blend. Save one combined `outputs/figures/pyramid_decomposition.png` arranging both pairs. This task exercises functions you already implemented in Tasks 4 and 5 — no new graded function is required.

In [8]:
decomposition_levels = 4
decomposition_pairs = [
    ("blend", "blend_left.png", "blend_right.png", "blend_mask.png"),
    ("blend3", "blend3_left.png", "blend3_right.png", "blend3_mask.png"),
]

fig, axes = a2_tools.axes_grid(6 * len(decomposition_pairs), decomposition_levels)
for pair_idx, (name, left_key, right_key, mask_key) in enumerate(decomposition_pairs):
    left = a2_tools.to_float01(images[left_key])
    right = a2_tools.to_float01(images[right_key])
    mask = a2_tools.to_float01(images[mask_key])
    if mask.ndim == 3:
        mask = mask[..., 0]

    gpyr_a = sc.gaussian_pyramid(left, decomposition_levels)
    gpyr_b = sc.gaussian_pyramid(right, decomposition_levels)
    lpyr_a = sc.laplacian_pyramid(left, decomposition_levels)
    lpyr_b = sc.laplacian_pyramid(right, decomposition_levels)
    mask_pyr = sc.gaussian_pyramid(mask, decomposition_levels)

    blended_levels = []
    for la, lb, m in zip(lpyr_a, lpyr_b, mask_pyr):
        mm = m[..., None] if la.ndim == 3 else m
        blended_levels.append((la * (1.0 - mm) + lb * mm).astype(np.float32))

    partial = np.asarray(blended_levels[-1], dtype=np.float32)
    progressive = [np.clip(partial, 0, 1)]
    for level in reversed(blended_levels[:-1]):
        partial = sc.expand(partial, level.shape[:2]) + level
        progressive.append(np.clip(partial, 0, 1))

    base = 6 * pair_idx
    for i in range(decomposition_levels):
        axes[base + 0, i].imshow(np.clip(gpyr_a[i], 0, 1)); axes[base + 0, i].set_title(f"{name} A gauss L{i}")
        axes[base + 1, i].imshow(rescale_laplacian(lpyr_a[i])); axes[base + 1, i].set_title(f"{name} A laplace L{i}")
        axes[base + 2, i].imshow(np.clip(gpyr_b[i], 0, 1)); axes[base + 2, i].set_title(f"{name} B gauss L{i}")
        axes[base + 3, i].imshow(rescale_laplacian(lpyr_b[i])); axes[base + 3, i].set_title(f"{name} B laplace L{i}")
        axes[base + 4, i].imshow(rescale_laplacian(blended_levels[i])); axes[base + 4, i].set_title(f"{name} blended laplace L{i}")
        recon_label = f"{name} recon step {i}"
        if i == 0:
            recon_label += " (coarsest)"
        elif i == decomposition_levels - 1:
            recon_label += " (finest)"
        axes[base + 5, i].imshow(np.clip(progressive[i], 0, 1)); axes[base + 5, i].set_title(recon_label)

for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()
fig.savefig(PATHS["figures"]/"pyramid_decomposition.png", dpi=150)
plt.close(fig)


**Decomposition analysis (2–4 sentences):** Explain what a Laplacian pyramid level represents, why blending per level instead of blending final pixel values directly avoids a visible seam, and connect this to the reconstruction check from Task 4.
Each laplacian level is the detail lst at one scaling operation. Blending happens separately at each level using the blurrier versions of the mask. Wider smoother areas blend gradually over a larger area while the sharper features blend right at the same as mentioned in response to task 5. Thats very different from splicing and leaving a high contrast line at the seam. The Task 4 check that summing all Laplacian levels reproduces the original to within 1.49e-08 confirms a blended pyramid reconstructs to a genuine, near-exact weighted combination of the two source images.

## 6. Results and Analysis

The immediate experiment responses contribute **10 analysis points collectively**. The synthesis contributes **3 points**. In at most 180 words, connect the most important filtering tradeoff, the strongest multi-scale finding, and one limitation. Do not restate each earlier observation.


**Synthesis:**
One of the tradeoffs across filtering and denoising is that average-based smoothing and impulse robust smoothing are no interchangeable. Bilateral filtering is strongest for gaussian noise and median filtering for impulse driven noise. The strongest multi-sclae finding was that laplacian reconstruction is essentially mathematically exact with max error found of 1.49e-8. So any visible failures in blendingcan be attributed to the mask or blending design rather than to the blending function itself. One limitation was the validity of the seam gradient metric used for blending which was not representative at all of visibly smoother blending.


## 7. Failure Analysis

### Task 7 — two controlled failures (5 points, counted under analysis and interpretation)

Show two reproducible failures from different topics. For each, identify the failure, show evidence, explain its mechanism, correct or mitigate it, and verify the improvement. The approved fallback is an excessively large sharpening amount that produces halos around strong edges, followed by a reduced or reformulated configuration. Students may instead use another reproducible failure. Save one coherent `outputs/figures/failure_analysis.png` and append `failure` rows to `experiment_metrics.csv` that quantify each failure and its mitigation.


In [9]:
edge_img = a2_tools.to_float01(images["test_pattern_gray.png"])

bad_amount = 8.0
good_amount = 1.0
sigma_sharp = 1.2
bad_sharp = sc.sharpen_image(edge_img, bad_amount, sigma_sharp)
good_sharp = sc.sharpen_image(edge_img, good_amount, sigma_sharp)
bad_overshoot = float(np.mean((bad_sharp > 1.0) | (bad_sharp < 0.0)))
good_overshoot = float(np.mean((good_sharp > 1.0) | (good_sharp < 0.0)))

append_metrics([
    {"experiment": "failure", "method": "sharpen_image", "condition": f"amount={bad_amount}", "metric": "out_of_range_fraction", "value": bad_overshoot, "units": "fraction"},
    {"experiment": "failure", "method": "sharpen_image", "condition": f"amount={good_amount}", "metric": "out_of_range_fraction", "value": good_overshoot, "units": "fraction"},
])

under_img = a2_tools.to_float01(images["exposure_under.png"])
bad_low, bad_high = 45.0, 55.0
good_low, good_high = 2.0, 98.0
bad_stretch = sc.normalize_contrast(under_img, bad_low, bad_high)
good_stretch = sc.normalize_contrast(under_img, good_low, good_high)
bad_clipped = float(np.mean((bad_stretch <= 0.0) | (bad_stretch >= 1.0)))
good_clipped = float(np.mean((good_stretch <= 0.0) | (good_stretch >= 1.0)))

append_metrics([
    {"experiment": "failure", "method": "normalize_contrast", "condition": f"low={bad_low}_high={bad_high}", "metric": "clipped_fraction", "value": bad_clipped, "units": "fraction"},
    {"experiment": "failure", "method": "normalize_contrast", "condition": f"low={good_low}_high={good_high}", "metric": "clipped_fraction", "value": good_clipped, "units": "fraction"},
])

fig, axes = a2_tools.axes_grid(2, 2)
axes[0, 0].imshow(np.clip(bad_sharp, 0, 1), cmap="gray"); axes[0, 0].set_title(f"sharpen amount={bad_amount} (halos)")
axes[0, 1].imshow(np.clip(good_sharp, 0, 1), cmap="gray"); axes[0, 1].set_title(f"sharpen amount={good_amount} (fixed)")
axes[1, 0].imshow(np.clip(bad_stretch, 0, 1)); axes[1, 0].set_title(f"normalize_contrast {bad_low}-{bad_high} (over-clipped)")
axes[1, 1].imshow(np.clip(good_stretch, 0, 1)); axes[1, 1].set_title(f"normalize_contrast {good_low}-{good_high} (fixed)")
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()
fig.savefig(PATHS["figures"]/"failure_analysis.png", dpi=150)
plt.close(fig)


**Failure analysis:** Applying sharpen_image to test_pattern_gray with amount = 8 and sigma=1.2 pushed 31.7% of the output pixels outside the valid range of [0, 1] producing bright and dark halo rings around hard edges. Unsharp masking with those values amplifies high frequencies 8x near any strong edges. Reducing amount helps but does not eliminate that problem.


## 8. Extension

### Task 8 — Extension (5 points)

Choose one: compare an approved advanced denoiser; alternative-color-space sharpening; local contrast enhancement; frequency-domain filtering; hybrid image; pyramid-mask comparison; small focus stack; or an approved enhancement. Submit one declared principal artifact under `outputs/extension/`, include one baseline comparison, and give a concise evidence-based explanation. Approved means anything you can build from the packages already imported in Section 2 (NumPy, OpenCV, scikit-image, Matplotlib, pandas). Use no additional packages or pretrained models.


In [10]:
color_img = a2_tools.to_float01(images["color_scene.png"])
amount_ext = 2.0
sigma_ext = 1.2

def sharpen_rgb_naive(image, amount, sigma):
    channels = [sc.sharpen_image(image[..., c], amount, sigma) for c in range(image.shape[2])]
    return np.stack(channels, axis=-1)

def sharpen_luminance(image, amount, sigma):
    ycrcb = cv2.cvtColor(image, cv2.COLOR_RGB2YCrCb)
    y, cr, cb = ycrcb[..., 0], ycrcb[..., 1], ycrcb[..., 2]
    y_sharp = sc.sharpen_image(y, amount, sigma)
    merged = np.stack([y_sharp, cr, cb], axis=-1).astype(np.float32)
    return cv2.cvtColor(merged, cv2.COLOR_YCrCb2RGB)

baseline = np.clip(sharpen_rgb_naive(color_img, amount_ext, sigma_ext), 0, 1)
extension_result = np.clip(sharpen_luminance(color_img, amount_ext, sigma_ext), 0, 1)

baseline_oob = float(np.mean((baseline <= 0.0) | (baseline >= 1.0)))
extension_oob = float(np.mean((extension_result <= 0.0) | (extension_result >= 1.0)))

append_metrics([
    {"experiment": "extension", "method": "sharpen_rgb_naive", "condition": f"amount={amount_ext}", "metric": "out_of_range_fraction", "value": baseline_oob, "units": "fraction"},
    {"experiment": "extension", "method": "sharpen_luminance", "condition": f"amount={amount_ext}", "metric": "out_of_range_fraction", "value": extension_oob, "units": "fraction"},
])

a2_tools.save_rgb(PATHS["extension"]/"luminance_sharpening.png", extension_result)

fig, axes = a2_tools.axes_grid(1, 3)
axes[0, 0].imshow(color_img); axes[0, 0].set_title("original")
axes[0, 1].imshow(baseline); axes[0, 1].set_title("naive RGB sharpen (baseline)")
axes[0, 2].imshow(extension_result); axes[0, 2].set_title("luminance-only sharpen")
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()
fig.savefig(PATHS["extension"]/"extension_comparison.png", dpi=150)
plt.close(fig)


**Extension question and conclusion (80–130 words total):**  I compared sharpening on the luminance channel over naive RGB sharpening to avoid artifacts.
Both versions look noticeably sharper. The brightness only sharpening pushed less pixels out of range (1.84% vs 1.92%). The difference is small mathematically but conceptually sharpening only brightness will never shift color values. Some of the effect is visible in the borders of the shapes being highly brighter than even the surrounding same color in RGB sharpening while the border on brightness only sharpening are more faithful to the original image.


## 9. Reflection and Reproducibility

Reproducibility, organization, and submission compliance are worth **10 points**: 4 for reproducible decisions and reflection, and 6 for correct execution, artifacts, schemas, disclosure, and packaging. In 100–150 words, identify one implementation decision another person must reproduce, one metric limitation, and one practice you will carry forward. Update the disclosure in Section 0.


**Reflection:** One implementation detail that someone would have to consider is the size of the sharpening blur kernel which uses 3 * sigma and varies between different library implementations. The seam gradient number for grading blending was unreliable as a metric as it suggests that pyramid blending is worse than plain splicing, which is not true when looking at them visually. Going forward I will try not to trust a single measure and instead rely on various metrics and visual inspection as well to catch those types of mismatches.


## 10. Submission Validation

Restart the kernel, run all cells, and export current HTML as `A2.html`. From any terminal directory, use the repository path:

```bash
python /path/to/common-setup/scripts/validate_submission.py --assignment A2
python /path/to/common-setup/scripts/package_submission.py --assignment A2
```

Submit the ASCII-safe `LastName_FirstName_A2.zip`. Do not submit PDF, datasets, caches, environments, checkpoints, or nested ZIP files.


### Canvas submission checklist

- [ ] The ZIP contains exactly one top-level folder named `LastName_FirstName_A2`.
- [ ] Section 0 identity fields (first name, last name, ABC123) are complete.
- [ ] `A2.ipynb` was restarted, run top to bottom, and saved without errors, and `A2.html` was exported after that final run.
- [ ] `src/` contains the required source files: `a2_tools.py` and `student_code.py`.
- [ ] `outputs/` contains only the required figures, tables, and the declared extension artifact — no extra files.
- [ ] `python /path/to/common-setup/scripts/validate_submission.py --assignment A2` reports zero errors; review any warnings.
- [ ] No datasets, environments, caches, checkpoints, hidden files, or temporary files are included.
- [ ] External resources and meaningful generative-AI use are disclosed in Section 0.
- [ ] A PDF export is not required and is not included.


In [11]:
for category in ("figures","tables","images","extension"): print(category,sorted(p.name for p in PATHS[category].glob("*") if p.is_file()))


figures ['.gitkeep', 'blending_comparison.png', 'contrast_enhancement.png', 'denoising_comparison.png', 'failure_analysis.png', 'filtering_results.png', 'pyramid_decomposition.png', 'pyramid_reconstruction.png', 'pyramid_visualization.png']
tables ['.gitkeep', 'experiment_metrics.csv', 'pyramid_metrics.json']
images ['.gitkeep']
extension ['.gitkeep', 'extension_comparison.png', 'luminance_sharpening.png']
